# CLASE 1 — Introducción al Testing Automatizado en Python con `pytest`

---



## Concepto clave: Testing

Testing es el proceso sistemático de verificar que el software cumple los comportamientos esperados bajo condiciones controladas.

No busca demostrar que el programa funciona.
Busca demostrar **dónde falla antes del usuario**.

---




# Niveles de testing como estrategia de control de riesgo en software

Un sistema falla por tres causas distintas:

1. La lógica está mal
2. Los componentes no se entienden
3. El proceso de negocio completo no funciona

Cada tipo de prueba nace para atacar **una categoría diferente de incertidumbre**.
Por eso no se sustituyen entre sí.

---

## 1. Testing Unitario — Validación semántica de la lógica del dominio

### Definición formal

Es la verificación aislada de una unidad de comportamiento determinista del sistema sin dependencias externas observables.

Unidad ≠ función necesariamente
Unidad = regla de negocio indivisible

---

### Qué intenta probar realmente

El unit test responde a una pregunta muy específica:

> Dado un conjunto de entradas válidas y controladas, ¿la regla del negocio produce exactamente la salida esperada?

Aquí no se valida infraestructura.
Se valida **significado**.

El objetivo es confirmar que el software representa correctamente la política del negocio.

---

### Qué tipo de defectos detecta

Errores semánticos:

* Fórmulas incorrectas
* Condiciones de frontera mal definidas
* Reglas contradictorias
* Casos no contemplados
* Validaciones incompletas

No detecta:

* errores de conexión
* problemas de concurrencia
* datos corruptos externos
* errores de serialización

---

### Ejemplo empresarial detallado — sistema de seguros

Regla:

> Un cliente paga recargo del 30% si tiene más de 2 siniestros en 12 meses
> pero si es cliente premium el recargo máximo es 15%

Implementación:



In [6]:
def calcular_recargo(siniestros, es_premium):
    if siniestros <= 2:
        return 0
    if es_premium:
        return 0.15
    return 0.30

Tests unitarios correctos:



In [ ]:
# Test: verifica que un cliente sin siniestros no tenga recargo
def test_cliente_sin_siniestros():
    # Arrange: preparamos los datos de entrada del escenario
    siniestros = 0          # cliente sin siniestros declarados
    es_premium = False      # no es cliente premium

    # Act: ejecutamos la función bajo prueba
    resultado = calcular_recargo(siniestros, es_premium)

    # Assert: verificamos que el comportamiento sea el esperado
    assert resultado == 0   # no debe haber recargo


# Test: cliente con alto riesgo (más de 2 siniestros) y no premium
def test_cliente_riesgoso():
    # Arrange: cliente con muchos siniestros y sin plan premium
    siniestros = 3          # más de 2 siniestros = alto riesgo
    es_premium = False      # cliente estándar

    # Act: ejecutamos la función
    resultado = calcular_recargo(siniestros, es_premium)

    # Assert: debe aplicar el recargo completo del 30%
    assert resultado == 0.30


# Test: cliente premium con muchos siniestros
def test_cliente_premium_limita_recargo():
    # Arrange: premium con muchos siniestros (el negocio limita su recargo)
    siniestros = 5          # varios siniestros
    es_premium = True       # es cliente premium

    # Act: ejecutamos la función
    resultado = calcular_recargo(siniestros, es_premium)

    # Assert: el recargo se limita al 15% para premium
    assert resultado == 0.15


# Función auxiliar para ejecutar los tests sin pytest (modo demostración)
def _ejecutar_pruebas_basicas():
    # Ejecuta manualmente cada función de prueba
    # Si alguna falla, Python lanzará AssertionError
    test_cliente_sin_siniestros()
    test_cliente_riesgoso()
    test_cliente_premium_limita_recargo()


# Función demostrativa para observar resultados en consola
def _demo():
    # Casos de ejemplo (entrada del sistema)
    ejemplos = [
        (0, False),   # cliente normal sin siniestros
        (3, False),   # cliente riesgoso
        (5, True),    # cliente premium con muchos siniestros
    ]

    # Recorre cada escenario
    for siniestros, es_premium in ejemplos:
        # Ejecuta la lógica del negocio
        recargo = calcular_recargo(siniestros, es_premium)

        # Muestra el resultado formateado con 2 decimales
        print(f"siniestros={siniestros}, premium={es_premium} -> recargo={recargo:.2f}")


# Punto de entrada del programa
if __name__ == "__main__":
    # Ejecuta primero las pruebas básicas
    _ejecutar_pruebas_basicas()

    # Luego muestra la simulación de uso real
    _demo()


Esto valida el **modelo del negocio**, no el sistema.

---

### Importancia arquitectónica

Los unit tests protegen el núcleo del sistema:

Dominio → inmutable
Infraestructura → cambiante

Si cambias BD, framework o API, el comportamiento del negocio debe permanecer idéntico.
Los unit tests garantizan esa estabilidad.

---

### Consecuencia práctica

Un sistema con buenos unit tests puede refactorizarse agresivamente sin riesgo lógico.

---

## 2. Testing de Integración — Validación de contratos entre componentes

### Definición formal

Verificación de que dos o más unidades independientes interactúan respetando su contrato técnico y semántico.

La palabra clave es **contrato**.

Un módulo no falla solo: falla cuando interpreta mal al otro.

---

### Qué intenta probar realmente

Responde a:

> ¿Los módulos intercambian información de forma correcta y compatible?

El problema aquí no es la lógica.
Es la traducción.

Software moderno = traductores:

* objetos → JSON
* JSON → SQL
* SQL → objetos
* servicio → servicio

---

### Tipo de defectos detectados

Errores de integración:

* columnas mal mapeadas
* tipos incompatibles
* encoding incorrecto
* pérdida de precisión
* fechas mal interpretadas
* transacciones incompletas
* errores de serialización

Todos estos errores pasan los unit tests.

---

### Ejemplo empresarial — API de pagos

Flujo:

Sistema envía:

```json
{
  "total": 15000,
  "currency": "COP"
}
```

Pasarela espera:

```json
{
  "amount": "15000.00",
  "currency_code": "COP"
}
```

La lógica de negocio es correcta.
El sistema falla igual.

El error es de integración semántica.

---

### Ejemplo con base de datos



---


## Beneficios empresariales de las pruebas

* Reduce costos de mantenimiento
* Permite refactorizar sin miedo
* Documenta comportamiento esperado
* Facilita trabajo en equipo


In [ ]:
# Clase que representa el repositorio de acceso a datos de Usuario
# (patrón Repository: separa la lógica del negocio de la base de datos)
class UsuarioRepo:

    # Método encargado de persistir un usuario en la base de datos
    def guardar(self, usuario):

        # Ejecuta una sentencia SQL usando un cursor activo de la conexión
        # "usuarios" es la tabla
        # "edad" es la columna donde se guardará el dato
        # %s es un placeholder del driver (evita inyección SQL)
        cursor.execute(
            "INSERT INTO usuarios (edad) VALUES (%s)",

            # Tupla de parámetros enviados al motor SQL
            # usuario.edad se toma del objeto de dominio Usuario
            # La coma es obligatoria para que Python lo interprete como tupla
            (usuario.edad,)
        )


Si la BD espera SMALLINT y Python envía string:

El unit test pasa
El sistema falla

Eso es un defecto de integración.

---

### Importancia arquitectónica

A medida que aumenta la distribución del sistema:

Monolito → pocos tests de integración
Microservicios → muchísimos tests de integración

Porque el mayor riesgo deja de ser la lógica y pasa a ser la comunicación.

---

### Consecuencia práctica

La mayoría de errores en producción real NO son de lógica.
Son de integración.

---

## 3. Testing End-to-End — Validación operacional del proceso de negocio

### Definición formal

Verificación automatizada del cumplimiento de un objetivo funcional observable desde la perspectiva del actor externo del sistema.

No verifica código
Verifica capacidad operativa

---

### Qué intenta probar realmente

Responde a:

> ¿El sistema permite completar el objetivo del usuario en condiciones reales?

La pregunta no es “¿funciona cada parte?”
Es:

> ¿la organización puede operar?

---

### Tipo de defectos detectados

Errores sistémicos:

* estados inconsistentes
* workflows incompletos
* pérdida de transacciones
* asincronía incorrecta
* errores de permisos cruzados
* orden incorrecto de eventos
* condiciones de carrera funcionales

---

### Ejemplo empresarial completo — reserva médica

Todo funciona individualmente:

* login correcto
* agenda disponible
* pago aprobado

Pero:

El turno no queda reservado.

El negocio falla
aunque el software no tenga errores aislados.

Eso solo lo detecta E2E.

---

### Naturaleza de esta prueba

El E2E valida propiedades emergentes:

El comportamiento del sistema completo no es la suma de los comportamientos individuales.

---

### Importancia organizacional

El E2E es la única prueba que responde:

> ¿Podemos salir a producción hoy?

No garantiza calidad interna
Garantiza operatividad externa

---

# Comparación estructural profunda

| Aspecto              | Unitario      | Integración    | End-to-End |
| -------------------- | ------------- | -------------- | ---------- |
| Nivel de abstracción | Regla         | Comunicación   | Operación  |
| Observador           | Desarrollador | Sistema        | Usuario    |
| Tipo de verdad       | Matemática    | Contractual    | Funcional  |
| Dependencias         | Ninguna       | Parciales      | Totales    |
| Velocidad            | ms            | seg            | min        |
| Costo de fallo       | Bajo          | Medio          | Crítico    |
| Propósito            | Exactitud     | Compatibilidad | Utilidad   |

---

# Idea clave para cerrar la explicación al estudiante

Los tres tipos prueban cosas diferentes:

* Unit test → el software piensa correctamente
* Integration test → el software habla correctamente
* E2E test → el software sirve correctamente

Eliminar cualquiera deja un tipo de ceguera.

Un sistema sin unit tests es frágil
sin integración es impredecible
sin E2E es inutilizable


# 2. Introducción a pytest

## Instalación

En terminal:

```bash
pip install pytest
```

Verificación:

```bash
pytest --version
```

---


## Primera prueba (live coding)

Crear archivo:

```
calculadora.py
```



In [ ]:
def sumar(a, b):
    return a + b


Crear:

```
test_calculadora.py
```



In [ ]:
from calculadora import sumar

def test_sumar():
    assert sumar(2, 3) == 5


Ejecutar:

```bash
pytest
```

---


### Explicación pedagógica importante

`pytest` busca automáticamente:

* archivos que empiecen por `test_`
* funciones que empiecen por `test_`

No necesita main
No necesita clases
No necesita framework complejo

---


## Qué es `assert`

Es la declaración de verdad del programa.

No pregunta:

> ¿Qué devuelve la función?

Pregunta:

> ¿Se comporta como el negocio exige?

---


# 3. Estructura correcta de pruebas

## Patrón AAA — obligatorio en la industria

Arrange – Act – Assert


In [ ]:
def test_sumar():
    # Arrange - Organizar
    a = 2
    b = 3

    # Act -Actuar
    resultado = sumar(a, b)

    # Assert - Afirmar
    assert resultado == 5


Explicar:

> Un test debe leerse como una historia corta.

---


## Mal test vs buen test

### Incorrecto

```python
def test_1():
    assert sumar(2,3)==5
```

### Correcto

```python
def test_sumar_dos_enteros_positivos():
    a = 2
    b = 3
    resultado = sumar(a, b)
    assert resultado == 5
```

El test también es documentación.

---


# 4. Manejo de excepciones

Modificar código:

```python
def dividir(a, b):
    if b == 0:
        raise ValueError("No se puede dividir por cero")
    return a / b
```

Test:

```python
import pytest
from calculadora import dividir

def test_dividir_por_cero():
    with pytest.raises(ValueError):
        dividir(10, 0)
```

---

Concepto clave:

> Un test exitoso también puede ser verificar que algo falle correctamente.

---


# 5. Setup y Teardown — preparación de entorno

Escenario real:
Sistema bancario necesita usuario antes de operar.

```python
import pytest

@pytest.fixture
def saldo_inicial():
    return 100
```

Uso:

```python
def test_retiro_valido(saldo_inicial):
    saldo = saldo_inicial - 40
    assert saldo == 60
```

---

Explicación:
Fixture = entorno controlado reproducible

Sin fixtures:

* tests dependen del orden
* fallan aleatoriamente
* imposibles de mantener

---


# 6. Organización profesional de tests

Estructura correcta de proyecto:

```
proyecto/
│
├── app/
│   └── calculadora.py
│
└── tests/
    └── test_calculadora.py
```

Ejecutar desde raíz:

```bash
pytest
```

---


## Organización por comportamiento

No agrupar por funciones
Agrupar por casos de negocio

Malo:

```
test_funciones.py
```

Correcto:

```
test_descuentos.py
test_usuarios.py
test_pagos.py
```

---


# 7. Ejercicio guiado

Sistema de comisiones:

`app/comisiones.py`


In [ ]:
def calcular_comision(ventas: float) -> float:
    if ventas < 0:
        raise ValueError("Ventas inválidas")
    if ventas < 1000:
        return 0
    if ventas < 5000:
        return ventas * 0.05
    return ventas * 0.1


### Necesidad

Escribir tests para:

1. ventas negativas
2. sin comisión
3. comisión media
4. comisión alta

---


#

# Solución:

Archivo: `test_comisiones/test_comisiones.py`



In [ ]:
import sys
from pathlib import Path

# Añade la raíz del proyecto a sys.path para importar app
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import pytest
from app.comisiones import calcular_comision


def test_ventas_negativas_lanzan_excepcion():
    # Arrange: ventas inválidas (negativas)
    ventas = -100

    # Act y Assert: la función debe lanzar ValueError
    with pytest.raises(ValueError):
        calcular_comision(ventas)


def test_ventas_sin_comision():
    # Arrange: ventas por debajo del umbral de 1000
    ventas = 800

    # Act: ejecutamos la función
    resultado = calcular_comision(ventas)

    # Assert: no hay comisión
    assert resultado == 0


def test_comision_media():
    # Arrange: ventas entre 1000 y 5000 (comisión 5%)
    ventas = 2000

    # Act: ejecutamos la función
    resultado = calcular_comision(ventas)

    # Assert: comisión del 5%
    assert resultado == ventas * 0.05


def test_comision_alta():
    # Arrange: ventas superiores a 5000 (comisión 10%)
    ventas = 6000

    # Act: ejecutamos la función
    resultado = calcular_comision(ventas)

    # Assert: comisión del 10%
    assert resultado == ventas * 0.10


---

## Cómo ejecutarlos

Desde la raíz del proyecto:

```bash
pytest -v
```

---


## Qué cubren realmente estos tests

| Test             | Riesgo cubierto              |
| ---------------- | ---------------------------- |
| ventas negativas | validación de entrada        |
| sin comisión     | frontera inferior de negocio |
| comisión media   | regla intermedia             |
| comisión alta    | regla superior               |

Esto garantiza que **cada tramo de la política comercial esté protegido contra regresiones**.


# Ejercicio práctico evaluativo — Testing de actividad de usuarios

## 1. Análisis de la regla (ANTES de escribir código)

Función dada:

`app/usuarios.py`



In [ ]:
def es_usuario_activo(ultimo_login_dias):
    return ultimo_login_dias <= 30

Interpretación funcional:

> Un usuario se considera activo si ha iniciado sesión en los últimos 30 días.

Esto implica:

| Categoría   | Resultado esperado |
| ----------- | ------------------ |
| 0 a 30 días | Activo             |
| > 30 días   | Inactivo           |

Pero como evaluadores debemos pensar más allá:

Un buen tester no prueba lo común.
Prueba donde el sistema puede romperse.

---

## 2. Identificación de riesgos

### Riesgos funcionales

* El límite exacto (30) puede fallar
* 31 podría ser considerado activo por error
* 0 podría fallar por comparación

### Riesgos de datos

* valores negativos
* valores extremadamente grandes

Esto define los casos obligatorios.

---

## 3. Diseño de casos de prueba

### A. Pruebas de frontera (Boundary testing)

Son las más importantes en lógica de negocio.

| Entrada | Resultado esperado |
| ------- | ------------------ |
| 29      | True               |
| 30      | True               |
| 31      | False              |

---

### B. Valores negativos

Un sistema real puede recibir datos corruptos.

| Entrada | Interpretación | Esperado                    |
| ------- | -------------- | --------------------------- |
| -1      | error de datos | True (según función actual) |

No validamos si está bien el negocio, validamos el comportamiento actual.

---

### C. Valores altos

| Entrada | Resultado esperado |
| ------- | ------------------ |
| 365     | False              |
| 9999    | False              |

---

# 4. Implementación con pytest

Archivo: `test_usuario_activo.py`



In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from app.usuarios import es_usuario_activo


# ---------------------------
# PRUEBAS DE LÍMITE
# ---------------------------

def test_usuario_activo_29_dias():
    # Arrange: usuario con login hace 29 días
    ultimo_login_dias = 29

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is True


def test_usuario_activo_30_dias():
    # Arrange: usuario con login exactamente hace 30 días
    ultimo_login_dias = 30

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is True


def test_usuario_inactivo_31_dias():
    # Arrange: usuario con login hace 31 días
    ultimo_login_dias = 31

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is False


# ---------------------------
# VALORES NEGATIVOS
# ---------------------------

def test_valor_negativo():
    # Arrange: valor inválido (negativo); el sistema actual lo considera activo
    ultimo_login_dias = -1

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is True


# ---------------------------
# VALORES ALTOS
# ---------------------------

def test_usuario_inactivo_un_ano():
    # Arrange: usuario sin login en 365 días
    ultimo_login_dias = 365

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is False


def test_usuario_inactivo_valor_extremo():
    # Arrange: valor muy alto
    ultimo_login_dias = 9999

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is False


---

## 5. Ejecución

```bash
pytest -v
```

---


## 6. Discusión:
Después de ejecutar los tests, se debe responder esta pregunta:

> ¿Un usuario con -1 días debería ser activo?

Concepto clave:

**Testing no solo detecta errores.
También descubre ambigüedades en los requerimientos.**

El test pasa, pero el sistema está mal definido.

---

## 7. Mejora propuesta

El dominio debería validar:

`app/usuarios.py`


In [ ]:
def es_usuario_activo(ultimo_login_dias: int) -> bool:
    if ultimo_login_dias < 0:
        raise ValueError("Valor inválido")
    return ultimo_login_dias <= 30

Y ahora los tests cambiarían.

Archivo: `test_usuario_activo.py`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import pytest
from app.usuarios import es_usuario_activo


# ---------------------------
# PRUEBAS DE LÍMITE (boundary)
# ---------------------------

def test_usuario_activo_0_dias():
    # Arrange: usuario que acaba de hacer login
    ultimo_login_dias = 0

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is True


def test_usuario_activo_29_dias():
    # Arrange: usuario con login hace 29 días
    ultimo_login_dias = 29

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is True


def test_usuario_activo_30_dias():
    # Arrange: usuario con login exactamente hace 30 días (límite inclusivo)
    ultimo_login_dias = 30

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is True


def test_usuario_inactivo_31_dias():
    # Arrange: usuario con login hace 31 días (ya inactivo)
    ultimo_login_dias = 31

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is False


# ---------------------------
# VALORES NEGATIVOS (excepción)
# ---------------------------

def test_valor_negativo_lanza_excepcion():
    # Arrange: valor inválido
    ultimo_login_dias = -1

    # Act y Assert: debe lanzar ValueError
    with pytest.raises(ValueError) as exc_info:
        es_usuario_activo(ultimo_login_dias)

    assert "Valor inválido" in str(exc_info.value)


# ---------------------------
# VALORES ALTOS
# ---------------------------

def test_usuario_inactivo_un_ano():
    # Arrange: usuario sin login en 365 días
    ultimo_login_dias = 365

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is False


def test_usuario_inactivo_valor_extremo():
    # Arrange: valor muy alto
    ultimo_login_dias = 9999

    # Act
    resultado = es_usuario_activo(ultimo_login_dias)

    # Assert
    assert resultado is False
